In [77]:
%config InlineBackend.figure_format = 'retina' # For high-resolution.
import pandas as pd
import numpy as np
import pydicom as dicom
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import os

import torch 
from torch import Tensor
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision import transforms, models

import cv2
from tqdm.notebook import tqdm


import warnings
warnings.filterwarnings('ignore')

In [66]:
def read_df(df_type):
    df_path = '/home/mel011/teams/dsc-180a---a14-[88137]/CANDID_PTX_csv/{}.csv'.format(df_type)
    df = pd.read_csv(df_path)[['Mask_Path', 'XRay_Path']]
    df['Mask_Path'] = df['Mask_Path'].str.replace('anw008', 'mel011')
    df['XRay_Path'] = df['XRay_Path'].str.replace('anw008', 'mel011')
    
    return df

In [71]:
train_df = read_df('train')
val_df = read_df('validation')
test_df = read_df('test')

In [40]:
test_mask = train_df.iloc[0, 0]
test_mask

'/home/mel011/teams/dsc-180a---a14-[88137]/CANDID_PTX_MASKS/0.0.00.400800.30.9.2.0.97200258210.2653758574681.8.png'

In [41]:
plt.imread(test_mask)[:, :, 0]

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)

In [81]:
## Hyperparameters
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)
print(torch.cuda.device_count())

RESOLUTION = 1024
BATCH_SIZE = 8
NUM_WORKERS = 4
PIN_MEMORY = True
DROP_LAST = True

# LEARNING_RATE = 
# NUM_EPOCHS = 



cuda
1


In [42]:
class CANDID_PTX(Dataset):
    def __init__(self, df, resolution):
        self.img_paths = df['XRay_Path'].values
        self.mask_paths = df['Mask_Path'].values
        
        self.resolution = resolution
          
        return
            
    
    def __len__(self):
        
        return self.img_paths.shape[0]
    
    
    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        img = dicom.dcmread(img_path).pixel_array
        img_min = np.min(img)
        img_max = np.max(img)
        img_norm = (img - img_min) / (img_max - img_min)
        img_norm = cv2.resize(img_norm, (self.resolution, self.resolution))
        img_norm = torch.tensor(img_norm).expand(3, self.resolution, self.resolution)
        
        mask_path = self.mask_paths[idx]
        mask = plt.imread(mask_path)[:, :, 0]
        mask = cv2.resize(mask, (self.resolution, self.resolution))
        mask = torch.tensor(mask).expand(3, self.resolution, self.resolution) 
        
        return img_norm, mask

In [72]:
train_ds = CANDID_PTX(train_df, RESOLUTION)
# val_ds = CANDID_PTX(val_df, 1024)
# test_ds = CANDID_PTX(test_df, 1024)

In [73]:
train_loader = DataLoader(train_ds, batch_size = BATCH_SIZE, num_workers = NUM_WORKERS, 
                          pin_memory = PIN_MEMORY, drop_last = DROP_LAST, shuffle = True)

# val_loader = DataLoader(val_ds, batch_size = BATCH_SIZE, num_workers = NUM_WORKERS, 
#                           pin_memory = PIN_MEMORY, drop_last = DROP_LAST, shuffle = False)

# test_loader = DataLoader(test_ds, batch_size = BATCH_SIZE, num_workers = NUM_WORKERS, 
#                           pin_memory = PIN_MEMORY, drop_last = DROP_LAST, shuffle = False)


In [78]:
test_loader = iter(train_loader)
test_sample = next(test_loader)

[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)
[W pthreadpool-cpp.cc:90] Warning: Leaking Caffe2 thread-pool after fork. (function pthreadpool)


In [79]:
len(test_sample)

2

In [80]:
len(test_sample[0])
test_sample[0].size()

torch.Size([8, 3, 512, 512])